In [ ]:
!pip install python-dotenv
!pip install ipython-genutils

import os, shutil, re
from dotenv import load_dotenv
from getpass import getpass

if not shutil.which("aria2c"):
    print("\n🟡aria2インストール")
    !{SUDO} apt update -qq
    !{SUDO} apt install -y aria2

if not shutil.which("mega-login"):
    print("\n🟡megacmdインストール")

    # /etc/os-release から OS 情報（ID, VERSION_ID）を取得
    os_info = {}
    with open("/etc/os-release") as f:
        for line in f:
            if "=" in line:
                key, val = line.strip().split("=", 1)
                os_info[key] = val.strip('"')

    os_id = os_info.get("ID", "ubuntu").lower()
    os_ver = os_info.get("VERSION_ID", "22.04")

# OSに応じたプレフィックスの割り当て
    if "debian" in os_id:
        prefix = f"Debian_{os_ver}"
    elif "ubuntu" in os_id:
        prefix = f"xUbuntu_{os_ver}"
    else:
        prefix = f"xUbuntu_22.04"

    deb_file = f"megacmd-{prefix}_amd64.deb"
    deb_url = f"https://mega.nz/linux/repo/{prefix}/amd64/{deb_file}"

    !wget -q -O {deb_file} {deb_url}
    !{SUDO} apt install -y ./{deb_file}
    !rm -f {deb_file}

load_dotenv(os.path.expanduser(f"{WORK_STEPS}/.env"))
email = os.getenv("MEGA_EMAIL")
password = os.getenv("MEGA_PASSWORD")

if not email or not password:
    raise RuntimeError("環境変数MEGA_EMAILかMEGA_PASSWORDが設定されていません。")

print("\n❗ MEGA ログインテストを実行します...")
!mega-login {email} {password} 2> /dev/null
!mega-whoami
!mega-logout

!rm -rf {HOME_DIR}/ImageGenSupporter
%cd {HOME_DIR}
!git clone --depth=1 https://github.com/forester3/ImageGenSupporter.git
print("\n✅ ImageGenSupporterのインストールが完了しました。")
print("\n✅ step0_initialize 終了")